# Mental High School

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import joblib
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
print('Libraries loaded successfully.')

In [ ]:
file_path = "/kaggle/input/datasets/damdung2005/mental-high-school-dataset/Mental School.csv"
df_raw = pd.read_csv(file_path)
print(f'Data shape: {df_raw.shape}')
df_raw.head()

In [ ]:
df = df_raw.copy()

# Normalize column names first to avoid metadata columns being missed because of leading spaces
df.columns = df.columns.str.strip()

# Convert whitespace-only strings to NaN, then try numeric conversion
for col in df.columns:
    if df[col].dtype == object:
        df[col] = df[col].apply(lambda x: np.nan if isinstance(x, str) and x.strip() == '' else x)
    try:
        df[col] = pd.to_numeric(df[col])
    except:
        pass

# Drop survey-design / metadata columns: these are useful for survey analysis, not predictive features
meta_cols = [
    'site', 'raceeth', 'q6orig', 'q7orig', 'record', 'orig_rec',
    'BMIPCT', 'weight', 'stratum', 'psu'
]
existing_meta_cols = [c for c in meta_cols if c in df.columns]
df.drop(columns=existing_meta_cols, inplace=True)

print(f'Dropped metadata/survey-design columns: {existing_meta_cols}')
print(f'Columns after cleaning: {df.shape[1]}')
print('Data types:', df.dtypes.value_counts().to_dict())

In [ ]:
QNUM_TO_ENGLISH = {
    1: 'Age', 2: 'Gender', 3: 'Grade', 4: 'Hispanic/Latino', 5: 'Race',
    6: 'Height (m)', 7: 'Weight (kg)',
    8: 'Seatbelt use', 9: 'Rode with drunk driver', 10: 'Drove after drinking',
    11: 'Texted while driving',
    12: 'Carried weapon to school', 13: 'Carried a gun', 14: 'Skipped school (felt unsafe)',
    15: 'Threatened/injured with weapon', 16: 'Physical fight (any)',
    17: 'Physical fight at school', 18: 'Witnessed neighbourhood violence',
    19: 'Forced sexual intercourse', 20: 'Other forced sex acts',
    21: 'Dating violence (forced sex)', 22: 'Dating violence (physical)',
    23: 'Discriminated (race/ethnicity)', 24: 'Bullied at school', 25: 'Cyberbullied',
    26: 'Felt sad or hopeless',
    27: 'Considered suicide', 28: 'Made suicide plan', 29: 'Attempted suicide',
    30: 'Suicide attempt needed treatment',
    31: 'Ever smoked cigarettes', 32: 'Smoked before age 13', 33: 'Current cigarette use',
    34: 'Cigarettes per day', 35: 'Ever used e-cigarettes', 36: 'Current e-cigarette use',
    37: 'Source of e-cigarettes', 38: 'Smokeless tobacco', 39: 'Cigars',
    40: 'Tried to quit smoking',
    41: 'Drank alcohol before 13', 42: 'Current alcohol use', 43: 'Binge drinking',
    44: 'Max drinks in a row', 45: 'Usual alcohol source', 46: 'Ever used marijuana',
    47: 'Marijuana initiation age', 48: 'Current marijuana use',
    49: 'Misused prescription painkillers', 50: 'Ever cocaine', 51: 'Ever inhalants',
    52: 'Ever heroin', 53: 'Ever methamphetamine', 54: 'Ever ecstasy',
    55: 'Ever injected drugs',
    56: 'Ever had sex', 57: 'First sex before 13', 58: 'Lifetime partners (4+)',
    59: 'Sexually active (3 mo)', 60: 'Alcohol/drugs before last sex',
    61: 'Condom use at last sex', 62: 'Birth control use',
    63: 'Sex of sexual partners', 64: 'Sexual orientation', 65: 'Transgender identity',
    66: 'Self-perceived weight', 67: 'Weight intention',
    68: '100% fruit juice', 69: 'Fruit intake', 70: 'Salad intake',
    71: 'Potato intake', 72: 'Carrot intake', 73: 'Other vegetables intake',
    74: 'Soda intake', 75: 'Breakfast frequency', 76: 'Physical activity (60+ min)',
    77: 'PE class attendance', 78: 'Sports team', 79: 'Sports concussion',
    80: 'Social media screen time', 81: 'Ever tested HIV', 82: 'Tested for STD',
    83: 'Last dental visit', 84: 'Poor mental health days',
    85: 'Hours of sleep', 86: 'Place of sleep', 87: 'Self-reported grades',
    88: 'Forced sex by adult (5+ years older)',
    89: 'Verbal abuse by adult at home', 90: 'Physical abuse by adult at home',
    91: 'Witnessed domestic violence', 92: 'Current misuse of painkillers',
    93: 'Ever hallucinogens', 94: 'Asked verbal consent before sex',
    95: 'Sports drinks intake', 96: 'Water intake', 97: 'Muscle strengthening',
    98: 'Sunburn frequency',
    99: 'Basic needs met by adult (food/clothing/safety)',
    100: 'Lived with substance abuser', 101: 'Lived with mentally ill/suicidal',
    102: 'Parent incarcerated',
    103: 'School connectedness', 104: 'Parents know whereabouts',
    105: 'Unfair discipline at school', 106: 'Difficulty concentrating/remembering',
    107: 'English proficiency',
}

def extract_qnum(col_name):
    s = col_name.strip()
    if s.startswith('q') and s[1:].isdigit():
        return int(s[1:])
    return None

col_to_display = {}
for c in df.columns:
    qnum = extract_qnum(c)
    if qnum and qnum in QNUM_TO_ENGLISH:
        col_to_display[c] = QNUM_TO_ENGLISH[qnum]
    else:
        col_to_display[c] = c.strip()

print('Example column mapping:')
for k, v in list(col_to_display.items())[:10]:
    print(f'{k} -> {v}')

In [ ]:
CLUSTER_MAP = {
    1: 'Demographics', 2: 'Demographics', 3: 'Demographics', 4: 'Demographics', 5: 'Demographics',
    6: 'Demographics', 7: 'Demographics',

    86: 'Housing & Basic Needs', 99: 'Housing & Basic Needs',
    104: 'Family Support & Monitoring',
    106: 'Functional Difficulty',

    23: 'School Climate & Academic Context', 24: 'School Climate & Academic Context',
    25: 'School Climate & Academic Context', 87: 'School Climate & Academic Context',
    103: 'School Climate & Academic Context', 105: 'School Climate & Academic Context',
    107: 'School Climate & Academic Context',

    66: 'Lifestyle Factors', 67: 'Lifestyle Factors', 68: 'Lifestyle Factors',
    69: 'Lifestyle Factors', 70: 'Lifestyle Factors', 71: 'Lifestyle Factors',
    72: 'Lifestyle Factors', 73: 'Lifestyle Factors', 74: 'Lifestyle Factors',
    75: 'Lifestyle Factors', 76: 'Lifestyle Factors', 77: 'Lifestyle Factors',
    78: 'Lifestyle Factors', 85: 'Lifestyle Factors', 95: 'Lifestyle Factors',
    96: 'Lifestyle Factors', 97: 'Lifestyle Factors',

    8: 'Safety & Driving', 9: 'Safety & Driving', 10: 'Safety & Driving', 11: 'Safety & Driving',

    12: 'Violence & Safety', 13: 'Violence & Safety', 14: 'Violence & Safety',
    15: 'Violence & Safety', 16: 'Violence & Safety', 17: 'Violence & Safety',
    18: 'Violence & Safety',

    19: 'Sexual Violence', 20: 'Sexual Violence', 21: 'Sexual Violence',
    22: 'Sexual Violence', 88: 'Sexual Violence',

    31: 'Substance Use', 32: 'Substance Use', 33: 'Substance Use', 34: 'Substance Use',
    35: 'Substance Use', 36: 'Substance Use', 37: 'Substance Use', 38: 'Substance Use',
    39: 'Substance Use', 40: 'Substance Use', 41: 'Substance Use', 42: 'Substance Use',
    43: 'Substance Use', 44: 'Substance Use', 45: 'Substance Use', 46: 'Substance Use',
    47: 'Substance Use', 48: 'Substance Use', 49: 'Substance Use', 50: 'Substance Use',
    51: 'Substance Use', 52: 'Substance Use', 53: 'Substance Use', 54: 'Substance Use',
    55: 'Substance Use', 92: 'Substance Use', 93: 'Substance Use',

    56: 'Sexual Behavior', 57: 'Sexual Behavior', 58: 'Sexual Behavior',
    59: 'Sexual Behavior', 60: 'Sexual Behavior', 61: 'Sexual Behavior',
    62: 'Sexual Behavior', 94: 'Sexual Behavior',

    63: 'Sexual Identity & Contacts', 64: 'Sexual Identity & Contacts',
    65: 'Sexual Identity & Contacts',

    89: 'Family & ACEs', 90: 'Family & ACEs', 91: 'Family & ACEs',
    100: 'Family & ACEs', 101: 'Family & ACEs', 102: 'Family & ACEs',

    27: 'Mental Health Indicators', 28: 'Mental Health Indicators',
    29: 'Mental Health Indicators', 30: 'Mental Health Indicators', 84: 'Mental Health Indicators',

    79: 'Other Health Risks', 98: 'Other Health Risks',
    80: 'Digital Behavior',
    81: 'Health Care & Preventive Services', 82: 'Health Care & Preventive Services',
    83: 'Health Care & Preventive Services',
}

col_to_cluster = {}
for c in df.columns:
    qnum = extract_qnum(c)
    if qnum and qnum in CLUSTER_MAP:
        col_to_cluster[c] = CLUSTER_MAP[qnum]
    else:
        col_to_cluster[c] = 'Other'

from collections import Counter
print('Columns per cluster:')
for cluster, count in Counter(col_to_cluster.values()).items():
    print(f'  {cluster}: {count}')

In [ ]:
# One-hot encode Q5 (race), because it is a "select one or more" question
q5_col = next((c for c in df.columns if extract_qnum(c) == 5), None)
if q5_col is not None and df[q5_col].dtype == object:
    race_values = df[q5_col].fillna('Missing').astype(str).str.strip()
    race_dummies = pd.get_dummies(race_values, prefix='q5_race', dtype=int)

    df = pd.concat([df.drop(columns=[q5_col]), race_dummies], axis=1)
    col_to_display.pop(q5_col, None)
    col_to_cluster.pop(q5_col, None)

    for dummy_col in race_dummies.columns:
        race_label = dummy_col.replace('q5_race_', '')
        col_to_display[dummy_col] = f'Race response: {race_label}'
        col_to_cluster[dummy_col] = 'Demographics'

    print(f'One-hot encoded {q5_col} into {len(race_dummies.columns)} race columns.')

# Encode any remaining categorical columns
categorical_cols = [c for c in df.columns if df[c].dtype == object]
print('Categorical columns to label-encode:', categorical_cols)

for c in categorical_cols:
    le = LabelEncoder()
    df[c] = df[c].fillna('Missing')
    df[c] = le.fit_transform(df[c].astype(str))
    if c in col_to_display:
        col_to_display[c] = col_to_display[c] + ' (encoded)'

print('Encoding completed.')

In [ ]:
q26_col = None
q84_col = None
for c in df.columns:
    qnum = extract_qnum(c)
    if qnum == 26:
        q26_col = c
    elif qnum == 84:
        q84_col = c

if not q26_col or not q84_col:
    raise ValueError('Target columns q26 or q84 not found.')

# Target definition:
# - Q26 = 1: felt sad or hopeless
# - Q84 >= 4: mental health was "most of the time" or "always" not good
df['Target'] = ((df[q26_col] == 1) | (df[q84_col] >= 4)).astype(int)
print('Target distribution:')
print(df['Target'].value_counts())

In [ ]:
def plot_distribution(series, title, color='#636EFA'):
    """Create a pie chart if ≤8 unique values, otherwise a histogram."""
    if series.nunique() <= 8:
        counts = series.value_counts().sort_index()
        fig = px.pie(values=counts.values, names=counts.index.astype(str),
                     title=title, hole=0.3, color_discrete_sequence=[color])
        fig.update_traces(textposition='inside', textinfo='percent+label')
    else:
        fig = px.histogram(series, nbins=20, title=title, color_discrete_sequence=[color])
        fig.update_layout(bargap=0.1)
    fig.update_layout(showlegend=False, height=350)
    return fig

# Select a few representative variables from each cluster (not all to avoid clutter)
cluster_representatives = {
    'Demographics': ['q1', 'q2', 'q3'],
    'Housing & Basic Needs': ['q86', 'q99'],
    'Family Support & Monitoring': ['q104'],
    'Functional Difficulty': ['q106'],
    'School Climate & Academic Context': ['q103', 'q87', 'q105', 'q23', 'q24'],
    'Lifestyle Factors': ['q85', 'q75', 'q76', 'q66', 'q68', 'q96'],
    'Violence & Safety': ['q12', 'q13', 'q14', 'q16', 'q18'],
    'Substance Use': ['q33', 'q42', 'q46', 'q47', 'q48'],
    'Sexual Behavior': ['q56', 'q58', 'q59'],
    'Sexual Identity & Contacts': ['q63', 'q64', 'q65'],
    'Family & ACEs': ['q89', 'q90', 'q100', 'q102'],
}

for cluster, cols in cluster_representatives.items():
    existing = [c for c in cols if c in df.columns]
    if not existing:
        continue
    print(f'\n--- {cluster} ---')
    n_cols = min(3, len(existing))
    fig = make_subplots(rows=1, cols=n_cols,
                        subplot_titles=[col_to_display.get(c, c) for c in existing[:n_cols]],
                        specs=[[{'type':'pie' if df[c].nunique() <= 8 else 'xy'} for c in existing[:n_cols]]])
    for i, col in enumerate(existing[:n_cols]):
        series = df[col].dropna()
        if series.nunique() <= 8:
            counts = series.value_counts().sort_index()
            fig.add_trace(go.Pie(labels=counts.index.astype(str), values=counts.values, showlegend=False),
                          row=1, col=i+1)
        else:
            fig.add_trace(go.Histogram(x=series, nbinsx=15, showlegend=False), row=1, col=i+1)
    fig.update_layout(title_text=cluster, height=400)
    fig.show()

In [ ]:
def qcol(num):
    for c in df.columns:
        if extract_qnum(c) == num:
            return c
    return None

# BMI (using q6, q7)
c6, c7 = qcol(6), qcol(7)
if c6 and c7:
    df['BMI'] = df[c7] / (df[c6] ** 2)
    col_to_cluster['BMI'] = 'Demographics'
    col_to_display['BMI'] = 'BMI (computed)'

# Academic Score:
# Q87 response 1-5 corresponds to Mostly A's -> Mostly F's.
# Codes 6=None of these grades and 7=Not sure are left as missing for this derived score.
c87 = qcol(87)
if c87:
    academic_map = {1: 5, 2: 4, 3: 3, 4: 2, 5: 1}
    df['Academic Score'] = df[c87].map(academic_map)
    col_to_cluster['Academic Score'] = 'School Climate & Academic Context'
    col_to_display['Academic Score'] = 'Academic Score (A=5 to F=1)'

# School Connectedness uses Q103 only
c103 = qcol(103)
if c103:
    df['School Connectedness'] = 6 - df[c103]
    col_to_cluster['School Connectedness'] = 'School Climate & Academic Context'
    col_to_display['School Connectedness'] = 'School Connectedness'

# Healthy Lifestyle Score
lifestyle_parts = []
c85 = qcol(85)
if c85:
    sleep_map = {4:1, 5:1, 3:0.7, 6:0.7, 2:0.4, 7:0.4, 1:0.1}
    lifestyle_parts.append(df[c85].map(sleep_map).fillna(0))
c75 = qcol(75)
if c75:
    lifestyle_parts.append((df[c75] - 1) / 7)
c76 = qcol(76)
if c76:
    lifestyle_parts.append((df[c76] - 1) / 7)
c96 = qcol(96)
if c96:
    lifestyle_parts.append((df[c96] - 1) / 6)
if lifestyle_parts:
    df['Healthy Lifestyle Score'] = sum(lifestyle_parts) / len(lifestyle_parts)
    col_to_cluster['Healthy Lifestyle Score'] = 'Lifestyle Factors'
    col_to_display['Healthy Lifestyle Score'] = 'Healthy Lifestyle Score'

# Substance Use Risk Score
sub_cols = [qcol(n) for n in [33, 36, 42, 48, 92]]
sub_cols = [c for c in sub_cols if c is not None]
if sub_cols:
    sub_scores = []
    for c in sub_cols:
        max_val = df[c].max()
        if max_val > 0:
            sub_scores.append(df[c].fillna(0) / max_val)
    if sub_scores:
        df['Substance Use Risk'] = sum(sub_scores) / len(sub_scores)
        col_to_cluster['Substance Use Risk'] = 'Substance Use'
        col_to_display['Substance Use Risk'] = 'Substance Use Risk Score'

# ACE Score (Adverse Childhood Experiences)
# Q89-Q91: higher than 1 means at least "rarely" experienced the adverse event.
# Q100-Q102: code 1 means "Yes".
ace_parts = []
for n in [89, 90, 91]:
    c = qcol(n)
    if c:
        ace_parts.append((df[c] > 1).astype(int))

for n in [100, 101, 102]:
    c = qcol(n)
    if c:
        ace_parts.append((df[c] == 1).astype(int))

if ace_parts:
    df['ACE Score'] = sum(ace_parts)
    col_to_cluster['ACE Score'] = 'Family & ACEs'
    col_to_display['ACE Score'] = 'ACE Score (0-6)'

print('New features created:')
for f in ['BMI', 'Academic Score', 'School Connectedness',
          'Healthy Lifestyle Score', 'Substance Use Risk', 'ACE Score']:
    if f in df.columns:
        print(f)

In [ ]:
derived = ['BMI', 'Academic Score', 'School Connectedness',
           'Healthy Lifestyle Score', 'Substance Use Risk', 'ACE Score']
present_derived = [f for f in derived if f in df.columns]
if present_derived:
    fig = make_subplots(rows=len(present_derived), cols=1,
                        subplot_titles=[col_to_display.get(f, f) for f in present_derived])
    for i, f in enumerate(present_derived):
        fig.add_trace(go.Box(x=df[f], name=f, marker_color='#FF7F0E'), row=i+1, col=1)
    fig.update_layout(height=250*len(present_derived), showlegend=False,
                      title='Distribution of Derived Features (Boxplots)')
    fig.show()

In [ ]:
exclude_nums = [26,27,28,29,30,84]   # mental health questions used to form target
exclude_cols = []
for c in df.columns:
    qnum = extract_qnum(c)
    if qnum in exclude_nums:
        exclude_cols.append(c)
exclude_cols.append('Target')

feature_cols = [c for c in df.columns if c not in exclude_cols
                and df[c].dtype in ['float64','int64']]
print(f'Number of features: {len(feature_cols)}')

# Impute missing values with mode
imputer = SimpleImputer(strategy='most_frequent')
X = imputer.fit_transform(df[feature_cols])
X = pd.DataFrame(X, columns=feature_cols)
y = df['Target']
print('X shape:', X.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2, random_state=42,
                                                    stratify=y)
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
print('Accuracy:', round(acc,4))
print(classification_report(y_test, y_pred))

importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

importance_df['cluster'] = importance_df['feature'].map(col_to_cluster).fillna('Other')
importance_df['display_name'] = importance_df['feature'].map(col_to_display).fillna(importance_df['feature'])

print('\nTop 15 features:')
print(importance_df[['feature','display_name','importance','cluster']].head(15).to_string(index=False))

In [ ]:
color_palette = {
    'Housing & Basic Needs': '#e74c3c',
    'Family Support & Monitoring': '#d35400',
    'Functional Difficulty': '#8e44ad',
    'School Climate & Academic Context': '#f1c40f',
    'Lifestyle Factors': '#2ecc71',
    'Demographics': '#3498db',
    'Violence & Safety': '#9b59b6',
    'Substance Use': '#e67e22',
    'Sexual Behavior': '#1abc9c',
    'Sexual Identity & Contacts': '#16a085',
    'Family & ACEs': '#34495e',
    'Safety & Driving': '#e84393',
    'Sexual Violence': '#c0392b',
    'Other Health Risks': '#7f8c8d',
    'Digital Behavior': '#2980b9',
    'Health Care & Preventive Services': '#27ae60',
    'Mental Health Indicators': '#bdc3c7',
    'Other': '#95a5a6'
}

# Top 20 overall
top20 = importance_df.head(20).sort_values('importance')
fig1 = px.bar(top20, x='importance', y='display_name', orientation='h',
              title='Top 20 Most Important Features',
              color='importance', color_continuous_scale='blues')
fig1.update_layout(height=600, yaxis_title='')
fig1.show()

# Total importance by cluster
cluster_imp = importance_df.groupby('cluster')['importance'].sum().reset_index()
fig2 = px.bar(cluster_imp, x='importance', y='cluster', orientation='h',
              title='Total Feature Importance by Cluster',
              color='cluster', color_discrete_map=color_palette)
fig2.update_layout(showlegend=False, yaxis_title='')
fig2.show()

# Importance for each important cluster
important_clusters = [
    'Housing & Basic Needs', 'Family Support & Monitoring', 'Functional Difficulty',
    'School Climate & Academic Context', 'Lifestyle Factors', 'Demographics',
    'Substance Use', 'Family & ACEs', 'Violence & Safety'
]
for cluster in important_clusters:
    cluster_df = importance_df[importance_df['cluster'] == cluster].sort_values('importance')
    if not cluster_df.empty:
        if cluster == 'Housing & Basic Needs':
            scale = 'reds'
        elif cluster == 'School Climate & Academic Context':
            scale = 'oranges'
        elif cluster == 'Lifestyle Factors':
            scale = 'greens'
        else:
            scale = 'purples'
        fig = px.bar(cluster_df, x='importance', y='display_name', orientation='h',
                     title=f'Feature Importance – {cluster}',
                     color='importance', color_continuous_scale=scale)
        fig.update_layout(yaxis_title='', height=250 + len(cluster_df)*20)
        fig.show()

In [ ]:
fig_cm = px.imshow(cm, text_auto=True,
                   x=['Healthy', 'At Risk'],
                   y=['Healthy', 'At Risk'],
                   color_continuous_scale='Blues',
                   title='Confusion Matrix')
fig_cm.show()
print(f'Accuracy: {acc:.4f}')
print(f'Precision (At Risk): {cm[1,1]/(cm[0,1]+cm[1,1]):.4f}')
print(f'Recall (At Risk): {cm[1,1]/(cm[1,0]+cm[1,1]):.4f}')

In [ ]:
joblib.dump(rf, 'mental_health_rf_model.pkl')
joblib.dump(imputer, 'imputer.pkl')
joblib.dump(col_to_cluster, 'col_to_cluster.pkl')
joblib.dump(col_to_display, 'col_to_display.pkl')
joblib.dump(feature_cols, 'feature_cols.pkl')
importance_df.to_csv('feature_importance.csv', index=False)
print('Model, imputer, and metadata saved.')
print('To use on new data:')
print('  1. Load model and imputer')
print('  2. Apply the same cleaning, encoding, and feature engineering')
print('  3. Use imputer.transform() and then rf.predict()')